### Minimal energy path of 2d system with two transition pathways

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import bisect

### Define the potential in 2d 

The potential has two local minimum states, connected by two transition pathways.

In [ ]:
def g(x, sigma=1):
    """Gaussian function
    """
    return np.exp(- 0.5 * (x / sigma)**2)

def dg(x, sigma=1):
    return - x * np.exp(-(x/sigma)**2) / sigma**2

class Potential2d:
    def __init__(self):
        self.dim = 2
        
    def V(self, X):
        # left and right minima
        u1 = g(X[1], 0.4) * (g(X[0] - 1, 0.3) + g(X[0] + 1, 0.3))
        # saddle points on paths
        u2 = g(X[0], 0.4) * (0.5 * g(X[1] - 1.2, 0.3) + g(X[1], 0.3))
        # barrier
        u3 = g(X[0], 0.4) * g(X[1]-0.5, 0.2)
       #upper path
        p1 = g(X[0]**2 + (X[1] - 1/3)**2 - 1.0, 0.5) / (1 + np.exp(4*(0.3-X[1])))    
        # lower path
        p2 = g(X[1], 0.2) * g(X[0], 0.8)   
        
        b1 = np.exp(5.0 *(-0.6 - X[1]))

        V = - 1.2 * u1 + 0.5 * u2 + 2 * u3 - p1 - p2 + b1 + 0.1 * (X[0] ** 2) + 0.1 * (X[1]** 2)
        return V
        
    def gradV(self, X):
        
        u1_dx = g(X[1], 0.4) * (dg(X[0] - 1, 0.3) + dg(X[0] + 1, 0.3))
        u1_dy = dg(X[1], 0.4) * (g(X[0] - 1, 0.3) + g(X[0] + 1, 0.3))
        
        u2_dx = dg(X[0], 0.4) * (0.8 * g(X[1] - 1.2, 0.3) + g(X[1], 0.3))
        u2_dy = g(X[0], 0.4) * (0.8 * dg(X[1] - 1.2, 0.3) + dg(X[1], 0.3))
        
        u3_dx = dg(X[0], 0.4) * g(X[1]-0.5, 0.2)
        u3_dy = g(X[0], 0.4) * dg(X[1]-0.5, 0.2)
        
        p1_dx = 2 * X[0] * dg(X[0]**2 + (X[1] - 1/3)**2 - 1.0, 0.5) / (1 + np.exp(4*(0.3-X[1])))

        p1_dy = 2 * (X[1] - 1/3) * dg(X[0]**2 + (X[1] - 1/3)**2 - 1.0, 0.5) / (1 + np.exp(4*(0.3-X[1]))) \
                + g(X[0]**2 + (X[1] - 1/3)**2 - 1.0, 0.5) \
                * 4 / (1 + np.exp(4*(0.3-X[1])))**2 * np.exp(4*(0.3-X[1]))

        p2_dx = g(X[1], 0.2) * dg(X[0], 0.8)
        p2_dy = dg(X[1], 0.2) * g(X[0], 0.8)
        
        b1_dx = 0
        b1_dy = -5.0 * np.exp(5.0 *(-0.6 - X[1]))

        V_dx = -1.2 * u1_dx + 0.5 * u2_dx + 2 * u3_dx - p1_dx - p2_dx + 0.2 * (X[0])       
        V_dy = -1.2 * u1_dy + 0.5 * u2_dy + 2 * u3_dy - p1_dy - p2_dy + b1_dy + 0.2 * (X[1])
        
        return np.array((V_dx, V_dy))
    
    def index_of_well(self,x, tol=0.2):
        w0 = np.array([-1,0])                
        w1 = np.array([1,0])                
        d0 = np.linalg.norm(x-w0, axis=1)
        d1 = np.linalg.norm(x-w1, axis=1)
        idx_0 = d0 < tol   
        idx_1 = d1 < tol   
        idx_vec = np.full(x.shape[0], -1, dtype=int)
        idx_vec[idx_0] = 0
        idx_vec[idx_1] = 1
        return idx_vec    
    
# sample the SDE using Euler-Maruyama scheme
def sample(pot, X0, beta=1.0, delta_t = 0.001, N=10000, seed=42):
    
    rng = np.random.default_rng(seed=seed)
     
    X = X0
    dim = 2 
    traj = []
    save = 100
    tlist = []
    for i in tqdm(range(N)):
        b = rng.normal(size=(dim,))
        X = X - pot.gradV(X) * delta_t + np.sqrt(2 * delta_t/beta) * b
        if i % save==0:
            traj.append(X)
            tlist.append(i * delta_t)

    return np.array(tlist), np.array(traj)    

###  Define an object of the potential class

In [ ]:
pot = Potential2d()

### Construct a 2d grid and evaluate the potential on the grid

In [ ]:
nx = 100
ny = 150

x_domain = [-1.8, 1.8]
y_domain = [-0.6, 1.8]
v_min_max = [-1.5, 0]
dx = (x_domain[1] - x_domain[0]) / nx
dy = (y_domain[1] - y_domain[0]) / ny

gridx = np.linspace(x_domain[0], x_domain[1], nx)
gridy = np.linspace(y_domain[0], y_domain[1], ny)
x_plot = np.outer(gridx, np.ones(ny)) 
y_plot = np.outer(gridy, np.ones(nx)).T 

# grid
x2d = np.concatenate((x_plot.reshape(nx * ny, 1), y_plot.reshape(nx * ny, 1)), axis=1)

# potential on the grid
pot_on_grid = np.array([pot.V(x) for x in x2d]).reshape(nx, ny)

### Visualize the potential

In [ ]:
fig, ax = plt.subplots()

im = ax.pcolormesh(x_plot, y_plot, pot_on_grid, cmap='coolwarm',shading='auto', vmin=v_min_max[0], vmax=v_min_max[1])
contours = ax.contour(x_plot, y_plot, pot_on_grid, 10, colors='black')

ax.clabel(contours, inline=True, fontsize=8)

fig.colorbar(im, ax=ax)
plt.show()

### Generate a long trajectory of the Brownian dynamics:
$$
dX_t = - \nabla V(X_t) dt + \sqrt{2\beta^{-1}} dW_t
$$

In [ ]:
# step-size
delta_t = 0.005
# initial state
x0 = np.array([-1.0,0])
# number of sampling steps
N = 1000000
# random seed
seed = 10
# coefficient in SDE. Large beta corresponds to small noise.
beta = 12

tlist, trajectory = sample(pot, x0, beta=beta, delta_t=delta_t, N=N, seed=seed)

print ('shape of the trajectory data:', trajectory.shape)

### Visualize the trajectory

**observation**:

1. Transitions are rare events.
2. Transitions may happen via both pathways.
3. The pathway on the top is more likely.

In [ ]:
### Plot the trajectory 
fig = plt.figure(figsize=(12,10))

ax0 = fig.add_subplot(2, 2, 1)
ax1 = fig.add_subplot(2, 2, 2)
ax2 = fig.add_subplot(2, 2, 3)
ax3 = fig.add_subplot(2, 2, 4)

nx = ny = 200
h = np.histogram2d(trajectory[:,0], trajectory[:,1], bins=[nx, ny], range=[[x_domain[0],x_domain[1]],[y_domain[0],y_domain[1]]])[0]
s = sum(sum(h))
im = ax0.imshow(h.T / (s * dx * dy), origin = "lower", \
                #h.T, origin = "lower", \
                extent=[x_domain[0],x_domain[1],y_domain[0], y_domain[1]], \
                cmap="viridis", vmin=0, vmax=0.2)
fig.colorbar(im, ax=ax0, shrink=0.7)
ax0.set_xlim([x_domain[0], x_domain[1]])
ax0.set_ylim([y_domain[0], y_domain[1]])
ax0.set_title('density')

#ax0.pcolormesh(x_plot, y_plot, pot_on_grid, cmap='coolwarm_r', shading='auto')
#ax1.scatter(trajectory[:,0], trajectory[:,1], marker='x')
ax1.plot(trajectory[:,0], trajectory[:,1])
ax1.set_xlim([x_domain[0], x_domain[1]])
ax1.set_ylim([y_domain[0], y_domain[1]])
ax1.set_title('trajectory')
ax1.set_aspect('equal')
#ax2.plot()
ax2.plot(range(len(trajectory[:,0])), trajectory[:,0])
ax2.set_ylim([x_domain[0], x_domain[1]])
ax2.set_title(r'$x$ along trajectory')

ax3.plot(range(len(trajectory[:,1])), trajectory[:,1])
ax3.set_ylim([y_domain[0], y_domain[1]])
ax3.set_title(r'$y$ along trajectory')
plt.show()

### Compute minimal energy path using string method

Besides two local minimal states, an initial path needs to be provided. 

In [ ]:
    
def find_MEP(pot, N, min_A, min_B, init_path, dt=0.001, NSteps=1000, with_reparam=True):
       
    delta_t = dt
    
    path = init_path

    for istep in range(NSteps):
        
        # Update: evolve each state on the path   
        for idx in range(N) :
            X = path[idx, :]
            path[idx,:] = X - pot.gradV(X) * delta_t 

        # Compute distances of adjacent states 
        dists = [0.0]
        for idx in range(N-1) :
            dists.append( np.linalg.norm(path[idx,:] - path[idx+1,:]) + dists[idx] )
        
        # normalize the total distance
        dists = dists / dists[-1]

        # Reparametrization

        new_states = [path[0,:]]
        
        for idx in range(N-2):
            r = (idx+1) / (N - 1)
            # find the index 
            pos = bisect.bisect_left(dists, r)
            # obtain new state by linear interpolation
            t = (r-dists[pos-1]) / (dists[pos] - dists[pos-1])                            
            new_states.append((1-t) * path[pos-1, :] + t * path[pos, :])

        new_states.append(path[-1,:])
        
        if with_reparam : 
            path = np.array(new_states)

    return path

### Identify the lower transition path 

In this test, the initial path is a straight (horizontal) line connecting the two local minimal states.

We iterate the path using string method for different number of steps. 

Since the lower transition pathway is close to horizontal. The initial path does not move significantly.

In [ ]:
# function to plot the potential as background
def plot_potential_on_axis(ax):
    
    # visualize the potential and its contour lines
    im = ax.pcolormesh(x_plot, y_plot, pot_on_grid, cmap='coolwarm', vmin=v_min_max[0], vmax=v_min_max[1])
    contours = ax.contour(x_plot, y_plot, pot_on_grid)

    ax.set_aspect('equal')
    ax.tick_params(axis='both', labelsize=10)

    ax.set_xticks([-1.5, -1.0, -0.5, 0, 0.5, 1.0])
    ax.set_yticks([-0.5, 0, 0.5, 1.0, 1.5, 2.0])
    ax.set_xlim([x_domain[0], x_domain[1]])
    ax.set_ylim([y_domain[0], y_domain[1]])    
        
fig = plt.figure(figsize=(12,6))

# two local minimal points a and b
xa = [-1.0, 0]
xb = [1.0, 0.0]

# initial path is a stright line 
init_path = path = np.linspace(xa, xb, N)

ax = fig.add_subplot(2, 3, 1)
plot_potential_on_axis(ax=ax)
# plot the initial path
ax.scatter(init_path[:,0], init_path[:,1], c='k')
ax.set_title('step 0')

# compute the path after 200 iterations.
path = find_MEP(pot, 30, xa, xb, init_path, NSteps=200)

ax = fig.add_subplot(2, 3, 2)
plot_potential_on_axis(ax=ax)

# plot the path
ax.scatter(path[:,0], path[:,1], c='k')
ax.set_title('step 200')

# compute the path after 400 iterations.
path = find_MEP(pot, 30, xa, xb, init_path, NSteps=400)

ax = fig.add_subplot(2, 3, 3)
plot_potential_on_axis(ax=ax)

# plot the path
ax.scatter(path[:,0], path[:,1], c='k')
ax.set_title('step 400')

# compute the path after 600 iterations.
path = find_MEP(pot, 30, xa, xb, init_path, NSteps=600)

ax = fig.add_subplot(2, 3, 4)
plot_potential_on_axis(ax=ax)

# plot the path
ax.scatter(path[:,0], path[:,1], c='k')
ax.set_title('step 600')

# compute the path after 800 iterations.
path = find_MEP(pot, 30, xa, xb, init_path, NSteps=800)

ax = fig.add_subplot(2, 3, 5)
plot_potential_on_axis(ax=ax)

# plot the path
ax.scatter(path[:,0], path[:,1], c='k')
ax.set_title('step 800')

# compute the path after 1000 iterations.
path = find_MEP(pot, 30, xa, xb, init_path, NSteps=1000)

ax = fig.add_subplot(2, 3, 6)
plot_potential_on_axis(ax=ax)

# plot the path
ax.scatter(path[:,0], path[:,1], c='k')
ax.set_title('step 1000')

plt.show()

# potential along the path
mep_1 = path
pot_on_mep_1 = [pot.V(state) for state in mep_1]

# plot the potential along the path
plt.plot(pot_on_mep_1)
plt.title('Potential along MEP 1')
plt.show()

### Identify the upper transition path 

In this test, we change the initial path in order to identify the second transition path. 

We iterate the path using string method for different number of steps. 

In [ ]:
fig = plt.figure(figsize=(12,6))

# two local minimal points a and b
xa = [-1.0, 0]
xb = [1.0, 0.0]

# initial path
init_path = np.concatenate((np.linspace(xa, [0,1.0], 15)[:-1], np.linspace([0,1.0], xb, 16)))

ax = fig.add_subplot(2, 3, 1)
plot_potential_on_axis(ax=ax)
# plot the path
ax.scatter(path[:,0], path[:,1], c='k')
ax.set_title('step 0')

# compute the path after 200 iterations.
path = find_MEP(pot, 30, xa, xb, init_path, NSteps=200)

ax = fig.add_subplot(2, 3, 2)
plot_potential_on_axis(ax=ax)

# plot the path
ax.scatter(path[:,0], path[:,1], c='k')
ax.set_title('step 200')

# compute the path after 400 iterations.
path = find_MEP(pot, 30, xa, xb, init_path, NSteps=400)

ax = fig.add_subplot(2, 3, 3)
plot_potential_on_axis(ax=ax)

ax.scatter(path[:,0], path[:,1], c='k')
ax.set_title('step 400')

# compute the path after 600 iterations.
path = find_MEP(pot, 30, xa, xb, init_path, NSteps=600)

ax = fig.add_subplot(2, 3, 4)
plot_potential_on_axis(ax=ax)
ax.scatter(path[:,0], path[:,1], c='k')
ax.set_title('step 600')

# compute the path after 800 iterations.
path = find_MEP(pot, 30, xa, xb, init_path, NSteps=800)

ax = fig.add_subplot(2, 3, 5)
plot_potential_on_axis(ax=ax)
ax.scatter(path[:,0], path[:,1], c='k')
ax.set_title('step 800')

# compute the path after 1000 iterations.
path = find_MEP(pot, 30, xa, xb, init_path, NSteps=1000)

ax = fig.add_subplot(2, 3, 6)
plot_potential_on_axis(ax=ax)
ax.scatter(path[:,0], path[:,1], c='k')
ax.set_title('step 1000')

plt.show()

# save the path
mep_2 = path

# potential along the path
pot_on_mep_2 = [pot.V(state) for state in mep_2]

# plot potential along the path
plt.plot(pot_on_mep_2)
plt.title('Potential along MEP 2')
plt.show()

### Let's compare the two transition pathways.

In [ ]:
fig = plt.figure(figsize=(10,5))

# plot the two paths in the figure on the left.
ax = fig.add_subplot(1, 2, 1)

plot_potential_on_axis(ax=ax)
ax.scatter(mep_1[:,0], mep_1[:,1], label='MEP 1')
ax.scatter(mep_2[:,0], mep_2[:,1], label='MEP 2')
ax.set_title('MEP 1 and 2')

# plot potential curves in the figure on the right.
ax = fig.add_subplot(1, 2, 2)

ax.plot(pot_on_mep_1, label='MEP 1')
ax.plot(pot_on_mep_2, label='MEP 2')
ax.set_title('Potential along MEP 1 and 2')
ax.legend()
plt.show()

### Let's compute the path again. 

We update the path using string method for 400 iterations. 

**We estimate**: 
1. the saddle poinnt by the state with the highest potential.
2. the unstable direction by finite difference.

It can be seen that the path has not converged yet. 

In [ ]:
# compute the path after 60 iterations.
path = find_MEP(pot, 30, xa, xb, init_path, NSteps=400)

fig = plt.figure(figsize=(10,5))
ax = fig.add_subplot(1, 2, 1)

plot_potential_on_axis(ax=ax)
ax.scatter(path[:,0], path[:,1], c='k')
ax.set_title('step 400')

# potential along the path
pot_on_line = [pot.V(state) for state in path]

# state with the highest potential
max_idx = np.argmax(pot_on_line)
X = path[max_idx,:]

# estimate tangent direction by finite difference 
tau = path[max_idx+1] - path[max_idx-1]
tau = tau / np.linalg.norm(tau)

# plot the state with highest potential
ax.plot(path[max_idx,0], path[max_idx,1], marker='o', c='w', markersize=12)

print ('tangent direction:', tau)

Check that the state with the highest potential is unstable under the ODE:
$$
\frac{dX_t}{dt} = - \nabla V(X_t)\,.
$$

In [ ]:
max_idx = np.argmax(pot_on_line)
X = path[max_idx,:]

X_list = [X]
N = 10000
delta_t = 0.001

# simulate the ODE for 1000 steps.
for idx in range(N) :
    X = X - pot.gradV(X) * delta_t 
    X_list.append(X)
    
fig, ax = plt.subplots()

plot_potential_on_axis(ax=ax)

X_traj = np.array(X_list)
# plot the trajectory of ODE 
ax.scatter(X_traj[::5,0], X_traj[::5,1])

ax.plot(X_traj[0,0], X_traj[0,1], marker='o', c='w', markersize=12)
ax.plot(X_traj[-1,0], X_traj[-1,1], marker='X', c='k', markersize=6)

plt.show()    

Refine the saddle point estimation using **Climbing Image Method**.

In [ ]:
# initial guess
X = path[max_idx,:]

X_list = [X]
N = 10000
delta_t = 0.001

for idx in range(N) :
    drift = pot.gradV(X)
    X = X + (- drift + 2.0 * np.dot(drift, tau) * tau) * delta_t
    X_list.append(X)

Verify that the dynamics converges to saddle point.

In [ ]:
fig, ax = plt.subplots()

plot_potential_on_axis(ax=ax)

X_traj = np.array(X_list)
# plot the trajectory of ODE 
ax.scatter(X_traj[::5,0], X_traj[::5,1])

ax.plot(X_traj[0,0], X_traj[0,1], marker='o', c='w', markersize=12)
ax.plot(X_traj[-1,0], X_traj[-1,1], marker='X', c='k', markersize=6)

plt.show()    